# BestModel: PCL Detection (DeBERTa-v3-base Ensemble + Threshold Tuning)

This notebook contains **what is needed to train and reproduce the best-performing model** used for submission:
- Fine-tunes **DeBERTa-v3-base** with **class weighting** and **oversampling** to ~30% PCL in training.
- Trains **two seeds** (42 and 123), ensembles by averaging probabilities.
- Tunes a **decision threshold** on the official dev set to maximise **PCL (positive-class) F1**.
- Writes `dev.txt` and `test.txt` in the repository root (one label per line: 0/1).

## Expected repo layout (minimum)
Place these files in your repo (paths can be changed in the config cell):
- `data/dontpatronizeme_pcl.tsv`
- `data/train_split.txt` (ids for train)  *(or edit loader to match your split format)*
- `data/dev_split.txt` (ids for dev)
- `data/test.tsv` (official test text; no labels)

If your data files differ, edit the **Config** cell only.


In [ ]:
# =====================
# Config (edit paths here)
# =====================
from pathlib import Path

REPO_ROOT = Path("..")
DATA_DIR = REPO_ROOT / "data"

TRAIN_TSV = DATA_DIR / "dontpatronizeme_pcl.tsv"
TEST_TSV  = DATA_DIR / "test.tsv"
TRAIN_IDS = DATA_DIR / "train_split.txt"
DEV_IDS   = DATA_DIR / "dev_split.txt"

# Output files required by spec (kept easy to find in repo root)
DEV_PRED_TXT  = REPO_ROOT / "dev.txt"
TEST_PRED_TXT = REPO_ROOT / "test.txt"

# Model / training
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 128

SEEDS = [42, 123]
EPOCHS = 6
LR = 2e-5
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

# Imbalance handling
POS_CLASS_WEIGHT = 9.5      # approx inverse prevalence from your analysis (~9.5% PCL)
TARGET_POS_FRAC = 0.30      # oversample PCL to ~30% of train

# Threshold tuning
THRESH_GRID = [x/100 for x in range(30, 71)]  # 0.30..0.70

ARTIFACT_DIR = Path(".") / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)

print("OK: config loaded")

In [ ]:
# ==========
# Dependencies
# ==========
import os, re, json, random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# =======================
# Data loading + preprocessing
# =======================
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"http\S+", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

def read_id_list(path: Path):
    ids = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                ids.append(line)
    return set(ids)

assert TRAIN_TSV.exists(), f"Missing {TRAIN_TSV}"
assert TRAIN_IDS.exists(), f"Missing {TRAIN_IDS}"
assert DEV_IDS.exists(), f"Missing {DEV_IDS}"
assert TEST_TSV.exists(), f"Missing {TEST_TSV}"

df = pd.read_csv(TRAIN_TSV, sep="\t", dtype=str)
# Expected columns in many versions of this dataset: 'par_id', 'text', 'label' (or similar).
col_map = {c.lower(): c for c in df.columns}
par_id_col = col_map.get("par_id") or col_map.get("id") or df.columns[0]
text_col   = col_map.get("text") or df.columns[1]
label_col  = col_map.get("label") or col_map.get("pcl") or df.columns[-1]

df = df[[par_id_col, text_col, label_col]].rename(columns={par_id_col:"id", text_col:"text", label_col:"label"})
df["text"] = df["text"].apply(clean_text)
df["label"] = df["label"].astype(int)

train_ids = read_id_list(TRAIN_IDS)
dev_ids   = read_id_list(DEV_IDS)

train_df = df[df["id"].astype(str).isin(train_ids)].copy()
dev_df   = df[df["id"].astype(str).isin(dev_ids)].copy()

assert len(train_df) > 0 and len(dev_df) > 0, "Train/dev split produced empty set. Check id files/columns."
print("train:", len(train_df), "dev:", len(dev_df))
print("train pos rate:", train_df["label"].mean(), "dev pos rate:", dev_df["label"].mean())

test_df = pd.read_csv(TEST_TSV, sep="\t", dtype=str)
test_col_map = {c.lower(): c for c in test_df.columns}
test_text_col = test_col_map.get("text") or test_df.columns[-1]
test_id_col = test_col_map.get("par_id") or test_col_map.get("id") or test_df.columns[0]
test_df = test_df[[test_id_col, test_text_col]].rename(columns={test_id_col:"id", test_text_col:"text"})
test_df["text"] = test_df["text"].apply(clean_text)

print("test:", len(test_df))

In [ ]:
# =======================
# Oversampling to target PCL fraction
# =======================
def oversample_to_fraction(df_in: pd.DataFrame, label_col="label", target_pos_frac=0.30, seed=42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    df_pos = df_in[df_in[label_col] == 1]
    df_neg = df_in[df_in[label_col] == 0]
    if len(df_pos) == 0 or len(df_neg) == 0:
        return df_in.copy()

    n_neg = len(df_neg)
    # want: n_pos / (n_pos + n_neg) ~= target_pos_frac -> n_pos ~= target_pos_frac/(1-target_pos_frac) * n_neg
    target_n_pos = int(round((target_pos_frac / (1 - target_pos_frac)) * n_neg))
    if target_n_pos <= len(df_pos):
        # downsample positives if needed (rare here)
        df_pos_new = df_pos.sample(n=target_n_pos, random_state=seed)
    else:
        # upsample positives with replacement
        idx = rng.choice(df_pos.index.values, size=target_n_pos, replace=True)
        df_pos_new = df_pos.loc[idx]
    df_out = pd.concat([df_neg, df_pos_new], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_out

train_os = oversample_to_fraction(train_df, target_pos_frac=TARGET_POS_FRAC, seed=SEEDS[0])
print("oversampled train:", len(train_os), "pos rate:", train_os["label"].mean())

In [ ]:
# =======================
# Torch datasets + tokenization
# =======================
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

dev_dataset  = TextDataset(dev_df["text"].tolist(), dev_df["label"].tolist())
test_dataset = TextDataset(test_df["text"].tolist(), None)

In [ ]:
# =======================
# Weighted loss via Trainer override
# =======================
class WeightedTrainer(Trainer):
    def __init__(self, pos_weight: float, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # weight tensor is for CrossEntropyLoss (class weights)
        self.class_weights = torch.tensor([1.0, float(pos_weight)], dtype=torch.float)

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**{k:v for k,v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    # positive-class F1 (PCL=1)
    f1_pos = f1_score(labels, preds, pos_label=1)
    prec, rec, _, _ = precision_recall_fscore_support(labels, preds, pos_label=1, average="binary", zero_division=0)
    return {"f1_pos": f1_pos, "precision_pos": prec, "recall_pos": rec}

In [ ]:
# =======================
# Train per seed
# =======================
def train_one_seed(seed: int):
    set_seed(seed)

    train_seed_df = oversample_to_fraction(train_df, target_pos_frac=TARGET_POS_FRAC, seed=seed)
    train_dataset = TextDataset(train_seed_df["text"].tolist(), train_seed_df["label"].tolist())

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    out_dir = ARTIFACT_DIR / f"seed_{seed}"
    args = TrainingArguments(
        output_dir=str(out_dir),
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_pos",
        greater_is_better=True,
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        pos_weight=POS_CLASS_WEIGHT,
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(str(out_dir / "best_model"))
    return trainer, metrics

trainers = {}
seed_metrics = {}

for s in SEEDS:
    tr, m = train_one_seed(s)
    trainers[s] = tr
    seed_metrics[s] = m
    print("seed", s, m)

with open(ARTIFACT_DIR / "seed_metrics.json", "w", encoding="utf-8") as f:
    json.dump(seed_metrics, f, indent=2)
print("saved:", ARTIFACT_DIR / "seed_metrics.json")

In [ ]:
# =======================
# Ensemble on dev: average probabilities
# =======================
def predict_proba(trainer: Trainer, dataset: Dataset):
    pred = trainer.predict(dataset)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return probs

dev_probs = []
test_probs = []

for s in SEEDS:
    dev_probs.append(predict_proba(trainers[s], dev_dataset))
    test_probs.append(predict_proba(trainers[s], test_dataset))

dev_probs_ens = np.mean(dev_probs, axis=0)
test_probs_ens = np.mean(test_probs, axis=0)

dev_y = dev_df["label"].values.astype(int)

def f1_at_threshold(probs, y_true, thr):
    y_pred = (probs[:,1] >= thr).astype(int)
    return f1_score(y_true, y_pred, pos_label=1)

best_thr, best_f1 = None, -1
for thr in THRESH_GRID:
    f1v = f1_at_threshold(dev_probs_ens, dev_y, thr)
    if f1v > best_f1:
        best_f1, best_thr = f1v, thr

print("best threshold:", best_thr, "best dev F1:", best_f1)

# Final dev report
dev_pred = (dev_probs_ens[:,1] >= best_thr).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(dev_y, dev_pred, pos_label=1, average="binary", zero_division=0)
cm = confusion_matrix(dev_y, dev_pred)
print("PCL precision:", prec, "recall:", rec, "f1:", f1)
print("confusion matrix:\n", cm)

with open(ARTIFACT_DIR / "ensemble_threshold.json", "w", encoding="utf-8") as f:
    json.dump({"seeds": SEEDS, "best_threshold": best_thr, "dev_f1_pos": float(best_f1)}, f, indent=2)
print("saved:", ARTIFACT_DIR / "ensemble_threshold.json")

In [ ]:
# =======================
# Write required prediction files (repo root): dev.txt and test.txt
# =======================
test_pred = (test_probs_ens[:,1] >= best_thr).astype(int)

DEV_PRED_TXT.parent.mkdir(parents=True, exist_ok=True)

with open(DEV_PRED_TXT, "w", encoding="utf-8") as f:
    for p in dev_pred.tolist():
        f.write(str(int(p)) + "\n")

with open(TEST_PRED_TXT, "w", encoding="utf-8") as f:
    for p in test_pred.tolist():
        f.write(str(int(p)) + "\n")

print("Wrote:", DEV_PRED_TXT, "and", TEST_PRED_TXT)
print("Dev positives:", int(dev_pred.sum()), "/", len(dev_pred))
print("Test positives:", int(test_pred.sum()), "/", len(test_pred))